# HF_TOKEN

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from termcolor import colored

# 1. Retrieve the token from Kaggle Secrets
user_secrets = UserSecretsClient()

try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print(colored(f"Success: Retrieved HF_TOKEN", "green"))
    
    # Corrected line below (it was 'ogin' before)
    login(token=hf_token)
    
except Exception as e:
    print(colored(f"Error: Could not retrieve HF_TOKEN. Make sure it is added in the 'Add-ons' -> 'Secrets' menu. Details: {e}", "red"))
    raise e

Success: Retrieved HF_TOKEN


# Working Example 1 

In [2]:
import torch
import time
import copy
import re
from termcolor import colored
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- CONFIGURATION ---
CONFIG = {
    "model_id": "meta-llama/Llama-3.2-3B-Instruct",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pruning_interval": 80,       # Trigger FREQUENTLY (every 80 tokens)
    "eviction_budget": 30,        # Evict 30 tokens per trigger
    "min_new_tokens": 500,        # [FORCE] Model CANNOT stop before this
    "max_new_tokens": 1500,
    "recent_window": 40,          # [SAFETY] Never prune the last 40 tokens
    "seed": 42
}

# The Summarization Prompt
SUMMARIZATION_PROMPT_TEXT = (
    "Time is up. Given the time I've spent and the approaches I've tried, "
    "I should stop thinking and now write summarization in one sentence.</think>"
)

# Markers for Segmentation
REASONING_MARKERS = [
    "Wait", "Alternatively", "Another angle", "But", "However", 
    "First", "Then", "Therefore", "Thus", "So", "Finally", 
    "Let's", "Let me", "Okay", "Hmm", "Note", "Consider",
    "Step", "To begin", "Now", "Next", "Consequently", "Check"
]

class KVCacheManager:
    """Safely handles Llama-3.2 DynamicCache vs Tuple formats."""
    @staticmethod
    def to_legacy(kv_cache):
        if hasattr(kv_cache, "to_legacy_cache"):
            return kv_cache.to_legacy_cache()
        return kv_cache 

    @staticmethod
    def from_legacy(legacy_cache, original_class):
        if hasattr(original_class, "from_legacy_cache"):
            return original_class.from_legacy_cache(legacy_cache)
        return legacy_cache

class TokenPruningEngine:
    def __init__(self, model, tokenizer, config):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.summ_tokens = tokenizer(SUMMARIZATION_PROMPT_TEXT, return_tensors="pt", add_special_tokens=False).input_ids.to(config['device'])

    def simple_sentence_split(self, text):
        """Replacement for NLTK: Splits on [.?!] followed by space."""
        return re.split(r'(?<=[.!?]) +', text)

    def segment_reasoning_chunks(self, text, total_generated_tokens):
        """Segments reasoning text into chunks for scoring."""
        sentences = self.simple_sentence_split(text)
        chunks = []
        current_chunk_text = []
        
        for sentence in sentences:
            is_start = any(sentence.strip().startswith(m) for m in REASONING_MARKERS)
            if is_start and current_chunk_text:
                chunks.append(" ".join(current_chunk_text))
                current_chunk_text = [sentence]
            else:
                current_chunk_text.append(sentence)
        if current_chunk_text:
            chunks.append(" ".join(current_chunk_text))

        if len(chunks) < 3: chunks = sentences 

        chunk_data = []
        current_idx = 0
        for i, chunk_str in enumerate(chunks):
            # Add space for correct tokenization estimation
            chunk_len = len(self.tokenizer.encode(" " + chunk_str, add_special_tokens=False))
            end_idx = min(current_idx + chunk_len, total_generated_tokens)
            if i == len(chunks) - 1: end_idx = total_generated_tokens 
            
            if end_idx > current_idx:
                chunk_data.append({
                    "id": i,
                    "text": chunk_str,
                    "indices": list(range(current_idx, end_idx))
                })
            current_idx = end_idx
        return chunk_data

    def get_importance_scores(self, past_key_values):
        """Injects prompt and calculates attention from </think>."""
        with torch.no_grad():
            outputs = self.model(
                input_ids=self.summ_tokens,
                past_key_values=past_key_values, 
                use_cache=True,
                output_attentions=True
            )
        
        all_layers_scores = []
        for layer_att in outputs.attentions:
            prompt_len = self.summ_tokens.shape[1]
            total_k_len = layer_att.shape[-1]
            reasoning_len = total_k_len - prompt_len
            
            # Scores: [heads, reasoning_len]
            scores = layer_att[0, :, -1, :reasoning_len]
            all_layers_scores.append(scores)
            
        return torch.stack(all_layers_scores)

    def hierarchical_eviction(self, kv_cache, attention_scores, chunks, prompt_offset):
        """Algorithm 1: Step-aware eviction."""
        cache_class = type(kv_cache)
        legacy_kv = KVCacheManager.to_legacy(kv_cache)
        num_layers = len(legacy_kv)
        
        # Determine strict bounds to avoid index errors
        total_gen_len = attention_scores.shape[-1]
        safe_boundary = total_gen_len - self.config['recent_window'] # Protect last N tokens
        
        pruned_kv_list = []
        evicted_text_log = []
        budget = self.config['eviction_budget']
        
        for layer_idx in range(num_layers):
            k_tensor, v_tensor = legacy_kv[layer_idx]
            layer_scores = attention_scores[layer_idx] 
            avg_head_scores = layer_scores.mean(dim=0) 
            
            # 1. Score Chunks
            chunk_metrics = []
            for chunk in chunks:
                valid_indices = [x for x in chunk['indices'] if x < safe_boundary]
                if not valid_indices: continue
                score = avg_head_scores[valid_indices].mean().item()
                chunk_metrics.append({"chunk": chunk, "score": score, "valid_indices": valid_indices})
            
            # 2. Sort by importance (ascending)
            sorted_chunks = sorted(chunk_metrics, key=lambda x: x['score'])
            
            # 3. Select Indices to Evict
            evict_indices_relative = []
            current_budget = budget
            
            for item in sorted_chunks:
                if current_budget <= 0: break
                n_evict = min(len(item['valid_indices']), current_budget)
                
                token_scores = [(idx, avg_head_scores[idx].item()) for idx in item['valid_indices']]
                token_scores.sort(key=lambda x: x[1])
                evict_targets = [x[0] for x in token_scores[:n_evict]]
                
                evict_indices_relative.extend(evict_targets)
                current_budget -= n_evict
                
                if layer_idx == 0 and n_evict > 0:
                    evicted_text_log.append(f"\"{item['chunk']['text'][:40]}...\" (Attn: {item['score']:.5f})")
            
            # 4. Apply Eviction
            evict_indices_absolute = [x + prompt_offset for x in evict_indices_relative]
            
            # --- FIX: Use prompt_offset, NOT prompt_len ---
            evict_indices_absolute = [x for x in evict_indices_absolute if x >= prompt_offset] 
            
            keep_mask = torch.ones(k_tensor.shape[2], dtype=torch.bool, device=k_tensor.device)
            keep_mask[evict_indices_absolute] = False
            
            new_k = k_tensor[:, :, keep_mask, :]
            new_v = v_tensor[:, :, keep_mask, :]
            pruned_kv_list.append((new_k, new_v))
            
        return KVCacheManager.from_legacy(pruned_kv_list, cache_class), evicted_text_log

def force_generate_and_prune(prompt, model, tokenizer, config):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(config['device'])
    prompt_len = input_ids.shape[1]
    
    pruner = TokenPruningEngine(model, tokenizer, config)
    past_key_values = None
    generated_tokens = []
    curr_input_ids = input_ids
    
    print(colored(f"Starting Forced Generation (Min: {config['min_new_tokens']} tokens)...", "cyan"))
    start_time = time.time()
    
    for step in range(config['max_new_tokens']):
        with torch.no_grad():
            outputs = model(
                input_ids=curr_input_ids,
                past_key_values=past_key_values,
                use_cache=True
            )
        
        logits = outputs.logits[:, -1, :]
        
        # --- FORCE CONTINUATION (NO EARLY EOS) ---
        if len(generated_tokens) < config['min_new_tokens']:
            logits[:, tokenizer.eos_token_id] = -float('inf')
            
        next_token = torch.argmax(logits, dim=-1).unsqueeze(0)
        
        if next_token.item() == tokenizer.eos_token_id:
            print(colored("\nEOS Reached (Allowed).", "green"))
            break
            
        past_key_values = outputs.past_key_values
        curr_input_ids = next_token
        generated_tokens.append(next_token.item())
        
        # --- PRUNING TRIGGER ---
        if len(generated_tokens) > (config['eviction_budget'] + config['recent_window']) and \
           len(generated_tokens) % config['pruning_interval'] == 0:
            
            print(colored(f"\n[Trigger @ Step {len(generated_tokens)}] Pruning...", "yellow"))
            
            # 1. Scores
            att_scores = pruner.get_importance_scores(past_key_values)
            
            # 2. Segment
            reasoning_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            chunks = pruner.segment_reasoning_chunks(reasoning_text, len(generated_tokens))
            
            # 3. Prune
            past_key_values, logs = pruner.hierarchical_eviction(
                past_key_values, att_scores, chunks, prompt_len
            )
            
            if logs:
                print(colored(f"  [Action] Evicted {config['eviction_budget']} tokens.", "red"))
                for log in logs[:2]: print(f"    - {log}")
            else:
                print("  [Action] Skipping (Context too short / protected).")

    total_time = time.time() - start_time
    full_output = tokenizer.decode(input_ids[0]) + tokenizer.decode(generated_tokens)
    return full_output, total_time

# --- EXECUTION ---
print(f"Loading {CONFIG['model_id']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_id'])
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_id'], 
    torch_dtype=torch.float16, 
    device_map="auto",
    attn_implementation="eager"
)

# "Paranoid Professor" Prompt
prompt_text = (
    "<|start_header_id|>system<|end_header_id|>\n\n"
    "You are a meticulous mathematician. Solve the problem by exploring multiple methods. "
    "Write out your internal monologue. If a method seems too simple, double check it. "
    "Be verbose. Do not stop until you are 100% sure.\n"
    "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
    "Three letters picked without replacement from qqqkkklkqkkk. Give prob of sequence qql.\n"
    "explicitly show steps too \n"
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

out, dur = force_generate_and_prune(prompt_text, model, tokenizer, CONFIG)

print("\n" + "="*50)
print("FINAL TEXT:")
print(out)

Loading meta-llama/Llama-3.2-3B-Instruct...


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Starting Forced Generation (Min: 500 tokens)...

[Trigger @ Step 80] Pruning...
  [Action] Evicted 30 tokens.
    - "Three letters picked without replacement..." (Attn: 0.00065)

[Trigger @ Step 160] Pruning...
  [Action] Evicted 30 tokens.
    - "The sequence of interest is Q-Q-L.

Let'..." (Attn: 0.00039)

[Trigger @ Step 240] Pruning...
  [Action] Evicted 30 tokens.
    - "The sequence of interest is Q-Q-L.

Let'..." (Attn: 0.00028)

[Trigger @ Step 320] Pruning...
  [Action] Evicted 30 tokens.
    - "The first Q...." (Attn: 0.00020)
    - "Picking the first Q...." (Attn: 0.00020)

[Trigger @ Step 400] Pruning...
  [Action] Evicted 30 tokens.
    - "The sequence of interest is Q-Q-L.

Let'..." (Attn: 0.00017)

[Trigger @ Step 480] Pruning...
  [Action] Evicted 30 tokens.
    - "There are 4 ways to pick the first Q.

2..." (Attn: 0.00016)
    - "1 way to pick the L...." (Attn: 0.00016)

EOS Reached (Allowed).

FINAL TEXT:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Y

# Working Example 2

In [2]:
import torch
import time
import copy
import re
from termcolor import colored
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- CONFIGURATION ---
CONFIG = {
    "model_id": "meta-llama/Llama-3.2-3B-Instruct",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pruning_interval": 80,       # Trigger FREQUENTLY (every 80 tokens)
    "eviction_budget": 30,        # Evict 30 tokens per trigger
    "min_new_tokens": 500,        # [FORCE] Model CANNOT stop before this
    "max_new_tokens": 1500,
    "recent_window": 40,          # [SAFETY] Never prune the last 40 tokens
    "seed": 42
}

# The Summarization Prompt
SUMMARIZATION_PROMPT_TEXT = (
    "Time is up. Given the time I've spent and the approaches I've tried, "
    "I should stop thinking and now write summarization in one sentence.</think>"
)

# Markers for Segmentation
REASONING_MARKERS = [
    "Wait", "Alternatively", "Another angle", "But", "However", 
    "First", "Then", "Therefore", "Thus", "So", "Finally", 
    "Let's", "Let me", "Okay", "Hmm", "Note", "Consider",
    "Step", "To begin", "Now", "Next", "Consequently", "Check"
]

class KVCacheManager:
    """Safely handles Llama-3.2 DynamicCache vs Tuple formats."""
    @staticmethod
    def to_legacy(kv_cache):
        if hasattr(kv_cache, "to_legacy_cache"):
            return kv_cache.to_legacy_cache()
        return kv_cache 

    @staticmethod
    def from_legacy(legacy_cache, original_class):
        if hasattr(original_class, "from_legacy_cache"):
            return original_class.from_legacy_cache(legacy_cache)
        return legacy_cache

class TokenPruningEngine:
    def __init__(self, model, tokenizer, config):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.summ_tokens = tokenizer(SUMMARIZATION_PROMPT_TEXT, return_tensors="pt", add_special_tokens=False).input_ids.to(config['device'])

    def simple_sentence_split(self, text):
        """Replacement for NLTK: Splits on [.?!] followed by space."""
        return re.split(r'(?<=[.!?]) +', text)

    def segment_reasoning_chunks(self, text, total_generated_tokens):
        """Segments reasoning text into chunks for scoring."""
        sentences = self.simple_sentence_split(text)
        chunks = []
        current_chunk_text = []
        
        for sentence in sentences:
            is_start = any(sentence.strip().startswith(m) for m in REASONING_MARKERS)
            if is_start and current_chunk_text:
                chunks.append(" ".join(current_chunk_text))
                current_chunk_text = [sentence]
            else:
                current_chunk_text.append(sentence)
        if current_chunk_text:
            chunks.append(" ".join(current_chunk_text))

        if len(chunks) < 3: chunks = sentences 

        chunk_data = []
        current_idx = 0
        for i, chunk_str in enumerate(chunks):
            # Add space for correct tokenization estimation
            chunk_len = len(self.tokenizer.encode(" " + chunk_str, add_special_tokens=False))
            end_idx = min(current_idx + chunk_len, total_generated_tokens)
            if i == len(chunks) - 1: end_idx = total_generated_tokens 
            
            if end_idx > current_idx:
                chunk_data.append({
                    "id": i,
                    "text": chunk_str,
                    "indices": list(range(current_idx, end_idx))
                })
            current_idx = end_idx
        return chunk_data

    def get_importance_scores(self, past_key_values):
        """Injects prompt and calculates attention from </think>."""
        with torch.no_grad():
            outputs = self.model(
                input_ids=self.summ_tokens,
                past_key_values=past_key_values, 
                use_cache=True,
                output_attentions=True
            )
        
        all_layers_scores = []
        for layer_att in outputs.attentions:
            prompt_len = self.summ_tokens.shape[1]
            total_k_len = layer_att.shape[-1]
            reasoning_len = total_k_len - prompt_len
            
            # Scores: [heads, reasoning_len]
            scores = layer_att[0, :, -1, :reasoning_len]
            all_layers_scores.append(scores)
            
        return torch.stack(all_layers_scores)

    def hierarchical_eviction(self, kv_cache, attention_scores, chunks, prompt_offset):
        """Algorithm 1: Step-aware eviction."""
        cache_class = type(kv_cache)
        legacy_kv = KVCacheManager.to_legacy(kv_cache)
        num_layers = len(legacy_kv)
        
        # Determine strict bounds to avoid index errors
        total_gen_len = attention_scores.shape[-1]
        safe_boundary = total_gen_len - self.config['recent_window'] # Protect last N tokens
        
        pruned_kv_list = []
        evicted_text_log = []
        budget = self.config['eviction_budget']
        
        for layer_idx in range(num_layers):
            k_tensor, v_tensor = legacy_kv[layer_idx]
            layer_scores = attention_scores[layer_idx] 
            avg_head_scores = layer_scores.mean(dim=0) 
            
            # 1. Score Chunks
            chunk_metrics = []
            for chunk in chunks:
                valid_indices = [x for x in chunk['indices'] if x < safe_boundary]
                if not valid_indices: continue
                score = avg_head_scores[valid_indices].mean().item()
                chunk_metrics.append({"chunk": chunk, "score": score, "valid_indices": valid_indices})
            
            # 2. Sort by importance (ascending)
            sorted_chunks = sorted(chunk_metrics, key=lambda x: x['score'])
            
            # 3. Select Indices to Evict
            evict_indices_relative = []
            current_budget = budget
            
            for item in sorted_chunks:
                if current_budget <= 0: break
                n_evict = min(len(item['valid_indices']), current_budget)
                
                token_scores = [(idx, avg_head_scores[idx].item()) for idx in item['valid_indices']]
                token_scores.sort(key=lambda x: x[1])
                evict_targets = [x[0] for x in token_scores[:n_evict]]
                
                evict_indices_relative.extend(evict_targets)
                current_budget -= n_evict
                
                if layer_idx == 0 and n_evict > 0:
                    evicted_text_log.append(f"\"{item['chunk']['text'][:40]}...\" (Attn: {item['score']:.5f})")
            
            # 4. Apply Eviction
            evict_indices_absolute = [x + prompt_offset for x in evict_indices_relative]
            
            # --- FIX: Use prompt_offset, NOT prompt_len ---
            evict_indices_absolute = [x for x in evict_indices_absolute if x >= prompt_offset] 
            
            keep_mask = torch.ones(k_tensor.shape[2], dtype=torch.bool, device=k_tensor.device)
            keep_mask[evict_indices_absolute] = False
            
            new_k = k_tensor[:, :, keep_mask, :]
            new_v = v_tensor[:, :, keep_mask, :]
            pruned_kv_list.append((new_k, new_v))
            
        return KVCacheManager.from_legacy(pruned_kv_list, cache_class), evicted_text_log

def force_generate_and_prune(prompt, model, tokenizer, config):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(config['device'])
    prompt_len = input_ids.shape[1]
    
    pruner = TokenPruningEngine(model, tokenizer, config)
    past_key_values = None
    generated_tokens = []
    curr_input_ids = input_ids
    
    print(colored(f"Starting Forced Generation (Min: {config['min_new_tokens']} tokens)...", "cyan"))
    start_time = time.time()
    
    for step in range(config['max_new_tokens']):
        with torch.no_grad():
            outputs = model(
                input_ids=curr_input_ids,
                past_key_values=past_key_values,
                use_cache=True
            )
        
        logits = outputs.logits[:, -1, :]
        
        # --- FORCE CONTINUATION (NO EARLY EOS) ---
        if len(generated_tokens) < config['min_new_tokens']:
            logits[:, tokenizer.eos_token_id] = -float('inf')
            
        next_token = torch.argmax(logits, dim=-1).unsqueeze(0)
        
        if next_token.item() == tokenizer.eos_token_id:
            print(colored("\nEOS Reached (Allowed).", "green"))
            break
            
        past_key_values = outputs.past_key_values
        curr_input_ids = next_token
        generated_tokens.append(next_token.item())
        
        # --- PRUNING TRIGGER ---
        if len(generated_tokens) > (config['eviction_budget'] + config['recent_window']) and \
           len(generated_tokens) % config['pruning_interval'] == 0:
            
            print(colored(f"\n[Trigger @ Step {len(generated_tokens)}] Pruning...", "yellow"))
            
            # 1. Scores
            att_scores = pruner.get_importance_scores(past_key_values)
            
            # 2. Segment
            reasoning_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            chunks = pruner.segment_reasoning_chunks(reasoning_text, len(generated_tokens))
            
            # 3. Prune
            past_key_values, logs = pruner.hierarchical_eviction(
                past_key_values, att_scores, chunks, prompt_len
            )
            
            if logs:
                print(colored(f"  [Action] Evicted {config['eviction_budget']} tokens.", "red"))
                for log in logs[:2]: print(f"    - {log}")
            else:
                print("  [Action] Skipping (Context too short / protected).")

    total_time = time.time() - start_time
    full_output = tokenizer.decode(input_ids[0]) + tokenizer.decode(generated_tokens)
    return full_output, total_time

# --- EXECUTION ---
print(f"Loading {CONFIG['model_id']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_id'])
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_id'], 
    torch_dtype=torch.float16, 
    device_map="auto",
    attn_implementation="eager"
)

# "Paranoid Professor" Prompt
prompt_text = (
    "<|start_header_id|>system<|end_header_id|>\n\n"
    "You are a meticulous mathematician. Solve the problem by exploring multiple methods. "
    "Write out your internal monologue. If a method seems too simple, double check it. "
    "Be verbose. Do not stop until you are 100% sure.\n"
    "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
    "Solve -42*r + 27*c = -1167 and 130*r + 4*c = 372 for r. \n"
    "explicitly show steps too \n"
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

out, dur = force_generate_and_prune(prompt_text, model, tokenizer, CONFIG)

print("\n" + "="*50)
print("FINAL TEXT:")
print(out)

Loading meta-llama/Llama-3.2-3B-Instruct...


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Starting Forced Generation (Min: 500 tokens)...

[Trigger @ Step 80] Pruning...
  [Action] Evicted 30 tokens.
    - "Let us embark on this mathematical odyss..." (Attn: 0.00063)

[Trigger @ Step 160] Pruning...
  [Action] Evicted 30 tokens.
    - "

Let me walk you through my thought pro..." (Attn: 0.00070)
    - "However, upon closer inspection, I reali..." (Attn: 0.00071)

[Trigger @ Step 240] Pruning...
  [Action] Evicted 30 tokens.
    - "

Let me walk you through my thought pro..." (Attn: 0.00048)
    - "However, upon closer inspection, I reali..." (Attn: 0.00061)

[Trigger @ Step 320] Pruning...
  [Action] Evicted 30 tokens.
    - "

Let me walk you through my thought pro..." (Attn: 0.00034)
    - "However, upon closer inspection, I reali..." (Attn: 0.00035)

[Trigger @ Step 400] Pruning...
  [Action] Evicted 30 tokens.
    - "However, upon closer inspection, I reali..." (Attn: 0.00027)

[Trigger @ Step 480] Pruning...
  [Action] Evicted 30 tokens.
    - "

Let me walk you throug